In [1]:
import pandas as pd
from datasets import load_dataset
import json
import os

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
print("Loading dataset...")

dataset = load_dataset(
    "ccdv/arxiv-summarization",
    split="train",
    streaming=True
)

print("Dataset loaded!")

Loading dataset...


README.md:   0%|          | 0.00/3.96k [00:00<?, ?B/s]

C:\Users\HI\Desktop\rag-arxiv-research\venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HI\.cache\huggingface\hub\datasets--ccdv--arxiv-summarization. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Dataset loaded!


In [5]:
papers = []
count = 0
target = 3000

print(f"Collecting {target} papers...")

for paper in dataset:
    article = paper.get('article', '')
    abstract = paper.get('abstract', '')
    
    if not abstract or len(abstract) < 100:
        continue
    
    papers.append({
        'id': str(count),
        'title': abstract[:80].strip(),
        'abstract': abstract.strip(),
    })
    count += 1
    
    if count % 500 == 0:
        print(f"Collected {count} papers...")
    
    if count >= target:
        break

print(f"Done! Collected {len(papers)} papers")

Collected 500 papers...
Collected 1000 papers...
Collected 1500 papers...
Collected 2000 papers...
Collected 2500 papers...
Collected 3000 papers...
Done! Collected 3000 papers


In [6]:
df = pd.DataFrame(papers)

print("Before cleaning:", len(df))

df = df.drop_duplicates(subset=['id'])
df = df[df['abstract'].str.len() > 100]
df = df[df['title'].str.len() > 10]
df = df.reset_index(drop=True)

print("After cleaning:", len(df))
print("\nSample abstract:")
print(df['abstract'][0][:200], "...")

Before cleaning: 3000
After cleaning: 3000

Sample abstract:
additive models play an important role in semiparametric statistics . 
 this paper gives learning rates for regularized kernel based methods for additive models . 
 these learning rates compare favour ...


In [7]:
os.makedirs('data', exist_ok=True)

df.to_csv('data/arxiv_ai_papers.csv', index=False)

print(f"Saved {len(df)} papers to data/arxiv_ai_papers.csv")
print(f"File size: {os.path.getsize('data/arxiv_ai_papers.csv') / 1024 / 1024:.2f} MB")

Saved 3000 papers to data/arxiv_ai_papers.csv
File size: 4.98 MB


In [8]:
df_check = pd.read_csv('data/arxiv_ai_papers.csv')

print("Dataset shape:", df_check.shape)
print("Columns:", df_check.columns.tolist())
print("\nFirst 3 abstracts preview:")
for i in range(3):
    print(f"\n{i+1}. {df_check['abstract'][i][:150]}...")

Dataset shape: (3000, 3)
Columns: ['id', 'title', 'abstract']

First 3 abstracts preview:

1. additive models play an important role in semiparametric statistics . 
 this paper gives learning rates for regularized kernel based methods for addit...

2. we have studied the leptonic decay @xmath0 , via the decay channel @xmath1 , using a sample of tagged @xmath2 decays collected near the @xmath3 peak p...

3. in 84 , 258 ( 2000 ) , mateos conjectured that current reversal in a classical deterministic ratchet is associated with bifurcations from chaotic to p...
